# Kaggle Submit — SmolVLM-500M LoRA adapter on test.csv 10k (T4 x2, resumable)

**Purpose:** run the finetuned adapter (`ShivRamSaud/smolvlm-500m-lora-astroclimb/adapter_best`, best eval_loss 0.30) over `/kaggle/input/competitions/astroclimb/test.csv` (10k rows) and write `submission.csv` (`id,same_figure,same_paper,related_papers,unrelated_papers` one-hot).

**Requirements:** `GPU T4 x2`, Internet ON, Secrets `HF_TOKEN` (read adapter + write progress/submission).

**Resumable by design:** every scored row streams to local staging jsonl; `submission/submission_progress.csv` (+raw) is re-uploaded to the SAME HF repo every 200 rows, and local staging is deleted at each phase end - **Hugging Face is the ONLY store**. A dead session loses nothing: done ids reload from HF and are skipped. ~10k rows x ~2-3s = ~6-9h total, plan ~2 sessions (8h time guard each).

**Train match (critical):** same SYSTEM_PROMPT/builders as finetune, captions `[:1000]`, `thumbnail(448)`, image-splitting OFF (single global tile), greedy 32 tokens.

**Test format:** OLD competition style — `obj_1`/`obj_2` cells hold caption text or base64 PNG, decoded per side in-cell. No folders, no UUIDs, no external images.


In [1]:
# --- 0. Config ---
MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
HF_REPO_ID = 'ShivRamSaud/smolvlm-500m-lora-astroclimb'
ADAPTER_SUBFOLDER = 'adapter_best'
TEST_CSV = '/kaggle/input/competitions/astroclimb/test.csv'
SUB_COLS = ['same_figure','same_paper','related_papers','unrelated_papers']
MAX_NEW_TOKENS = 32
CAP_CHARS = 1000
MAX_SEQ_LEN = 2048
HF_PUSH_EVERY = 200
TIME_BUDGET = 8*3600
OUT_DIR = '/kaggle/working/smolvlm_submit'
PROG_LOCAL = OUT_DIR + '/submission_progress.csv'
RAW_LOCAL = OUT_DIR + '/raw_test.jsonl'
HF_PROG = 'submission/submission_progress.csv'
HF_RAW = 'submission/raw_test.jsonl'
HF_FINAL = 'submission/submission.csv'
import os; os.makedirs(OUT_DIR, exist_ok=True)
print('adapter=' + HF_REPO_ID + '/' + ADAPTER_SUBFOLDER)
print('results -> ' + HF_REPO_ID + '/submission/ (same repo)')
print('TEST=' + TEST_CSV)


adapter=ShivRamSaud/smolvlm-500m-lora-astroclimb/adapter_best
results -> ShivRamSaud/smolvlm-500m-lora-astroclimb/submission/ (same repo)
TEST=/kaggle/input/competitions/astroclimb/test.csv


In [2]:
# --- 1. Setup (Kaggle T4 x2: versions checked BEFORE import, never reimport torch) ---
import os, json, time, re, gc, base64, sys, subprocess
from pathlib import Path
from io import BytesIO
import pandas as pd, numpy as np
from PIL import Image
from tqdm import tqdm
import importlib.metadata as _im
from packaging import version as _pv
import importlib as _il
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
try:
    _drv = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True).strip().splitlines()[0]
    _drv_major = int(_drv.split('.')[0])
except Exception:
    _drv_major = 0
print('nvidia driver major: ' + str(_drv_major))
try:
    _torch_v = _im.version('torch')
except Exception: _torch_v = '0.0.0'
print('installed torch ' + _torch_v)
if _pv.parse(_torch_v) < _pv.parse('2.5.0'):
    _cu = 'cu121' if _drv_major >= 525 else 'cu118'
    print('Upgrading torch to 2.5.1+' + _cu + ' ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'torch==2.5.1', 'torchaudio==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/' + _cu])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] in ('torch', 'torchvision', 'torchaudio', 'transformers', 'peft', 'accelerate', 'typing_extensions'): del sys.modules[m]
    _il.invalidate_caches()
import torch
print('torch ' + torch.__version__ + ' cuda ' + str(torch.cuda.is_available()))
assert torch.cuda.is_available(), 'Enable GPU: Settings -> Accelerator -> GPU T4 x2'
assert torch.cuda.device_count() >= 2, 'Need 2xT4'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
_need = _pv.parse(_im.version('transformers')) < _pv.parse('4.48.0')
print('transformers ' + _im.version('transformers') + ' need>=4.48: ' + str(_need))
if _need:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.48', 'accelerate', 'peft'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] in ('transformers', 'peft', 'accelerate', 'typing_extensions'): del sys.modules[m]
    _il.invalidate_caches()
import transformers; print('transformers ' + transformers.__version__)
import peft; print('peft ' + peft.__version__)
try:
    import torchao as _ta
    print('stale torchao ' + _ta.__version__ + ' present - removing (breaks peft, unused here)...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
    for m in list(sys.modules.keys()):
        if m.split('.')[0] == 'torchao': del sys.modules[m]
    print('torchao removed')
except ImportError:
    print('no torchao present (fine)')


nvidia driver major: 580
installed torch 2.0.0
Upgrading torch to 2.5.1+cu121 ...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
cuml 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
cudf 23.8.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.3 which is incompatible.
cudf 23.8.0 requires protobuf<5,>=4.21, but you have protobuf 3.20.3 which is incompatible.
cuml 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
cuml 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires dask==2023.7.1, b

torch 2.5.1+cu121 cuda True
['Tesla T4', 'Tesla T4']
transformers 4.36.0 need>=4.48: True


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cudf 23.8.0 requires cupy-cuda11x>=12.0.0, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.7 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 11.0.0 which is incompatible.
dask-cuda 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
dask-cuda 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cuda 23.8.0 requires pandas<1.6.0dev0,>=1.3, but you have pandas 2.0.3 which is incompatible.
dask-cudf 23.8.0 requires dask==2023.7.1, but you have dask 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires distributed==2023.7.1, but you have distributed 2023.12.0 which is incompatible.
dask-cudf 23.8.0 requires pandas<1.6.0dev0,>=1.3, but 

transformers 5.16.1


/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


peft 0.20.0
no torchao present (fine)


In [3]:
# --- 1b. HF login (read adapter + write progress/submission) ---
from huggingface_hub import login
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e: print('No HF_TOKEN: ' + str(e))
assert HF_TOKEN, 'Add HF_TOKEN (write) to Kaggle Secrets'
login(token=HF_TOKEN)
print('HF login OK -> ' + HF_REPO_ID)


HF login OK -> ShivRamSaud/smolvlm-500m-lora-astroclimb


In [4]:
# --- 2. Test data (10k) + format detect + FULL decode audit ---
assert Path(TEST_CSV).exists(), 'test missing: ' + TEST_CSV
test = pd.read_csv(TEST_CSV)
print('test rows=' + str(len(test)))
ID_COL = 'id' if 'id' in test.columns else test.columns[0]
print('id column: ' + str(ID_COL))
OLD_FMT = {'obj_1','obj_2'} <= set(test.columns)
print('OLD-format cols (base64 in-cell, no folders): ' + str(OLD_FMT))
assert OLD_FMT, 'test.csv must have obj_1/obj_2 columns!'
assert 'label' not in test.columns, 'test must be unlabeled - the model generates the labels!'
def is_image_str(s):
    if not isinstance(s, str) or len(s) < 200: return False
    return s.strip().startswith('iVBORw0KGgo')
test['_o1_img'] = test['obj_1'].apply(is_image_str)
test['_o2_img'] = test['obj_2'].apply(is_image_str)
print(pd.crosstab(test['_o1_img'], test['_o2_img']))
_n_img_ok=_n_img_bad=_n_txt=_n_empty=0; _bad=[]
for _i, _r in test.iterrows():
    for _cc in ['obj_1','obj_2']:
        _s=_r[_cc]
        if pd.isna(_s) or (isinstance(_s,str) and len(_s.strip())==0):
            _n_empty+=1; _bad.append((_i,_cc,'empty'))
        elif is_image_str(_s):
            try:
                _d=Image.open(BytesIO(base64.b64decode(_s))).convert('RGB'); _d.load(); _n_img_ok+=1; del _d
            except Exception as _e:
                _n_img_bad+=1; _bad.append((_i,_cc,'decode_fail:'+str(_e)[:100]))
        else: _n_txt+=1
print('cells=' + str(len(test)*2) + ' images_ok=' + str(_n_img_ok) + ' images_bad=' + str(_n_img_bad) + ' captions=' + str(_n_txt) + ' empty=' + str(_n_empty))
assert _n_img_bad==0, 'UNDECODABLE images'
assert _n_img_ok+_n_txt+_n_empty==2*len(test)==20000 and len(test)==10000, 'row/cell count mismatch!'
print('FULL DECODE AUDIT OK - 10k rows accounted')
test.drop(columns=['_o1_img','_o2_img'], inplace=True)


test rows=10000
id column: id
OLD-format cols (base64 in-cell, no folders): True
_o2_img  False  True 
_o1_img              
False     3000   4000
True         0   3000
cells=20000 images_ok=10000 images_bad=0 captions=10000 empty=0
FULL DECODE AUDIT OK - 10k rows accounted


In [5]:
# --- 3. Builders (train-exact prompt contract, OLD-format test reader) ---
SYSTEM_PROMPT = '''You are an expert in astrophysics figures and captions. Given Object A and Object B (each is either a figure image or a caption text), classify their relation into exactly ONE label based ONLY on what you see/read - no DOI or metadata is provided.\n
\n
Classes:\n
- same_figure: The caption directly describes the figure in front of you. Visual elements (axes, labels, numbers, morphology) are mentioned verbatim in the text, or the text reads like \"Figure X shows...\" matching the image.\n
- same_paper: Same study, different figures. Similar writing style, same instruments/datasets/authors hinted in text, or visual style (fonts, colors, layout) is consistent, but NOT a direct caption-figure match.\n
- related_papers: Different papers where one builds on the other. Overlapping methods, shared datasets, or a figure/caption that looks like a cited prior result, but style/authors differ.\n
- unrelated_papers: No clear link. Different topics, instruments, scales, or writing/visual style with no overlap.\n
\n
Base your decision only on visual and textual content. Do not assume same_figure is impossible for any pair type - judge from alignment.\n
Output ONLY the lowercase label (e.g., related_papers), no explanation, no punctuation.\n
'''
def side_old(row, col, name):
    s=row[col]
    if pd.isna(s) or (isinstance(s, str) and len(s.strip()) == 0):
        return {'name':name,'is_img':False,'text':'','missing':True}
    if is_image_str(s):
        try: im=Image.open(BytesIO(base64.b64decode(s))).convert('RGB')
        except: return {'name':name,'is_img':False,'text':'','missing':True}
        return {'name':name,'is_img':True,'pil':im,'missing':False}
    return {'name':name,'is_img':False,'text':str(s)[:CAP_CHARS],'missing':False}
def build_messages_test(row):
    parts=[side_old(row,'obj_1','Object A'), side_old(row,'obj_2','Object B')]
    if any(p.get('missing') for p in parts): return None
    content=[{'type':'text','text': SYSTEM_PROMPT+'\n'}]
    for p in parts:
        if p['is_img']:
            im=p['pil'].copy(); im.thumbnail((448,448))
            content.append({'type':'image','image':im})
        if p['is_img']:
            content.append({'type':'text','text':'\n'+p['name']+': [Figure image]'})
        else:
            content.append({'type':'text','text':'\n'+p['name']+' (caption): '+p['text']})
    content.append({'type':'text','text':'\nAnswer with one label:'})
    return [{'role':'user','content':content}]
import re
label_pattern = re.compile('(same_figure|same_paper|related_papers|unrelated_papers)', re.IGNORECASE)
def parse_label(text):
    m=label_pattern.search(str(text).lower()); return m.group(1).lower() if m else 'unrelated_papers'
_d0=build_messages_test(test.iloc[0])
print('demo built:', _d0 is not None, '| parts:', len(_d0[0]['content']) if _d0 else 0)


demo built: True | parts: 5


In [6]:
# --- 4. Model: base bf16 + adapter_best from HF (train-match vision caps) ---
from transformers import AutoProcessor
try:
    from transformers import AutoModelForImageTextToText as ModelClass; print('Using AutoModelForImageTextToText')
except:
    from transformers import AutoModel as ModelClass; print('Using AutoModel fallback')
try:
    from peft import PeftModel
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'peft'])
    from peft import PeftModel
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
print('Processor: ' + type(processor).__name__)
try:
    _ip = processor.image_processor
    if hasattr(_ip, 'do_image_splitting'): _ip.do_image_splitting = False
    print('image_splitting OFF -> single global tile per image (train match)')
except Exception as _e: print('vision override skipped: ' + str(_e)[:200])
try:
    _tk = processor.tokenizer
    if _tk.pad_token is None:
        _tk.pad_token = _tk.eos_token
        print('pad_token set to eos')
    print('pad_token id=' + str(_tk.pad_token_id))
except Exception as _e: print('pad fix skipped: ' + str(_e)[:200])
base = ModelClass.from_pretrained(MODEL_ID, device_map='auto', max_memory={0:'14GB',1:'14GB'}, torch_dtype=torch.bfloat16, trust_remote_code=True)
print('Base bf16: ' + type(base).__name__)
model = PeftModel.from_pretrained(base, HF_REPO_ID, subfolder=ADAPTER_SUBFOLDER)
print('Adapter attached: ' + str(model.active_adapter) + ' (from ' + HF_REPO_ID + '/' + ADAPTER_SUBFOLDER + ')')
model.eval()
print('Has generate: ' + str(hasattr(model, 'generate')))
try: _DEV = model.device
except: _DEV = torch.device('cuda:0')
for _i in [0,1]:
    _r=test.iloc[_i]; _m=build_messages_test(_r)
    assert _m is not None, 'demo row failed to build!'
    _t=processor.apply_chat_template(_m, tokenize=False, add_generation_prompt=True)
    _im=[p['image'] for t in _m for p in t['content'] if p.get('type')=='image'] or None
    _in=processor(text=[_t], images=_im, padding=True, return_tensors='pt')
    _in={k:(v.to(_DEV) if hasattr(v,'to') else v) for k,v in _in.items()}
    if 'pixel_values' in _in: _in['pixel_values']=_in['pixel_values'].to(torch.bfloat16)
    _L=_in['input_ids'].shape[1]
    with torch.no_grad(): _o=model.generate(**_in, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    _dec=processor.batch_decode(_o[:,_L:], skip_special_tokens=True)[0]
    print('sample ' + str(_i) + ' id=' + str(_r[ID_COL]) + ' raw=' + repr(_dec.strip()[:60]) + ' -> ' + parse_label(_dec))
    del _in,_o; torch.cuda.empty_cache()
print('Generate check OK')


Using AutoModelForImageTextToText


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

...

Base bf16: Idefics3ForConditionalGeneration


adapter_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

adapter_best/adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 76.6MB            

adapter_best/adapter_model.safetensors: downloading bytes:           |  0.00B            

Adapter attached: default (from ShivRamSaud/smolvlm-500m-lora-astroclimb/adapter_best)
Has generate: True
sample 0 id=0 raw='same_paper.' -> same_paper
sample 1 id=1 raw='same_paper.' -> same_paper
Generate check OK


In [7]:
# --- 5. Inference 10k: resume from HF progress, stream local + push HF ---
import time, json
from tqdm import tqdm
from huggingface_hub import HfApi, hf_hub_download
api=HfApi(); api.create_repo(repo_id=HF_REPO_ID, private=True, exist_ok=True)
def _hf_push(local, remote, msg, tries=4):
    import time as _t
    for _a in range(tries):
        try:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=HF_REPO_ID, commit_message=msg)
            return True
        except Exception as e:
            print('push ' + remote + ' attempt ' + str(_a+1) + '/' + str(tries) + ' failed: ' + str(e)[:160])
            _t.sleep(45)
    return False
T_END=time.time()+TIME_BUDGET
done={}
for _ra in range(4):
    try:
        _pp=hf_hub_download(repo_id=HF_REPO_ID, filename=HF_PROG, repo_type='model')
        _pd=pd.read_csv(_pp)
        done=dict(zip(_pd['id'].astype(str), _pd['pred']))
        print('resumed ' + str(len(done)) + ' rows from HF ' + HF_PROG)
        break
    except Exception as e:
        import time as _t2
        if '404' in str(e) or 'Entry Not Found' in str(e):
            print('no HF progress file yet - starting fresh')
            break
        print('progress download attempt ' + str(_ra+1) + '/4 failed: ' + str(e)[:160])
        _t2.sleep(30)
else:
    print('WARNING: could not reach HF progress file - starting fresh; already-pushed rows will be RESCORED (wasteful but harmless)')
preds=dict(done)
skipped=[]; n_oom=0; n_parse=0; t0=time.time(); n_new=0
_rawf=open(RAW_LOCAL,'a')
for idx in tqdm(range(len(test))):
    if time.time()>T_END: print('TIME GUARD at ' + str(idx) + '/' + str(len(test)) + ' - stopping clean, progress safe'); break
    row=test.iloc[idx]; _id=str(row[ID_COL])
    if _id in preds: continue
    msgs=build_messages_test(row)
    if msgs is None: skipped.append({'idx':int(idx),'id':_id,'reason':'unbuildable_input'}); preds[_id]='unrelated_papers'; continue
    try:
        text=processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        images=[p['image'] for t in msgs for p in t['content'] if p.get('type')=='image'] or None
        inputs=processor(text=[text], images=images, padding=True, return_tensors='pt')
    except Exception as _e:
        print('encode failed idx ' + str(idx) + ', fallback: ' + str(_e)[:160])
        inputs=processor(text=['Object A: undecodable. Object B: undecodable. Answer with one label:'], padding=True, return_tensors='pt')
    inputs={k:(v.to(_DEV) if hasattr(v,'to') else v) for k,v in inputs.items()}
    if 'pixel_values' in inputs: inputs['pixel_values']=inputs['pixel_values'].to(torch.bfloat16)
    L=inputs['input_ids'].shape[1]
    try:
        with torch.no_grad(): out=model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    except RuntimeError as e:
        if 'out of memory' not in str(e).lower(): raise
        torch.cuda.empty_cache(); gc.collect()
        try:
            with torch.no_grad(): out=model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        except: decoded=''; n_oom+=1; pred='unrelated_papers'
        else: decoded=processor.batch_decode(out[:,L:], skip_special_tokens=True)[0]; pred=parse_label(decoded)
    else: decoded=processor.batch_decode(out[:,L:], skip_special_tokens=True)[0]; pred=parse_label(decoded)
    if pred=='unrelated_papers' and 'unrelated_papers' not in str(decoded).lower(): n_parse+=1
    preds[_id]=pred; n_new+=1
    _rawf.write(json.dumps({'id':_id,'pred':pred,'raw':decoded})+'\n'); _rawf.flush()
    del inputs
    if idx%50==0: torch.cuda.empty_cache()
    if n_new%HF_PUSH_EVERY==0:
        pd.DataFrame([{'id':k,'pred':v} for k,v in preds.items()]).to_csv(PROG_LOCAL,index=False)
        _ok1=_hf_push(PROG_LOCAL, HF_PROG, 'progress ' + str(len(preds)) + '/' + str(len(test)))
        _ok2=_hf_push(RAW_LOCAL, HF_RAW, 'raw ' + str(len(preds)) + '/' + str(len(test)))
        print('progress ' + str(len(preds)) + '/' + str(len(test)) + ' pushed=' + str(_ok1 and _ok2) + ' elapsed ' + str(round((time.time()-t0)/3600,2)) + 'h', flush=True)
_rawf.close()
pd.DataFrame([{'id':k,'pred':v} for k,v in preds.items()]).to_csv(PROG_LOCAL,index=False)
_okf1=_hf_push(PROG_LOCAL, HF_PROG, 'progress final ' + str(len(preds)) + '/' + str(len(test)))
_okf2=_hf_push(RAW_LOCAL, HF_RAW, 'raw final loop-end')
if _okf1 and _okf2:
    for _f in [PROG_LOCAL, RAW_LOCAL]:
        try: os.remove(_f)
        except: pass
    print('final progress pushed; local staging deleted - HF is source of truth')
else:
    print('FINAL PUSH INCOMPLETE - local staging kept at ' + OUT_DIR + '; download it from Kaggle output before closing the session!')
print('LOOP DONE scored_total=' + str(len(preds)) + '/' + str(len(test)) + ' new_this_session=' + str(n_new) + ' skipped=' + str(len(skipped)) + ' oom=' + str(n_oom) + ' parse_fb=' + str(n_parse) + ' elapsed=' + str(round((time.time()-t0)/3600,2)) + 'h')


no HF progress file yet - starting fresh


  2%|▏         | 199/10000 [02:56<2:20:26,  1.16it/s]

progress 200/10000 pushed=True elapsed 0.05h


  4%|▍         | 399/10000 [05:58<2:35:08,  1.03it/s]

progress 400/10000 pushed=True elapsed 0.1h


...

progress 9800/10000 pushed=True elapsed 2.53h


100%|█████████▉| 9999/10000 [2:34:15<00:00,  1.42it/s]

progress 10000/10000 pushed=True elapsed 2.57h


100%|██████████| 10000/10000 [2:34:17<00:00,  1.08it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


final progress pushed; local staging deleted - HF is source of truth
LOOP DONE scored_total=10000/10000 new_this_session=10000 skipped=0 oom=0 parse_fb=0 elapsed=2.57h


In [8]:
# --- 6. Assemble submission.csv (all 10k ids, one-hot, test order) ---
from huggingface_hub import HfApi, hf_hub_download
api=HfApi()
try:
    _pp=hf_hub_download(repo_id=HF_REPO_ID, filename=HF_PROG, repo_type='model')
    _scored=pd.read_csv(_pp); print('scored rows from HF: ' + str(len(_scored)))
except Exception as e:
    _scored=pd.read_csv(PROG_LOCAL)
    print('HF download failed (' + str(e)[:120] + ') - local staging fallback: ' + str(len(_scored)))
preds=dict(zip(_scored['id'].astype(str), _scored['pred']))
full=pd.DataFrame({'id':test[ID_COL].astype(str)})
full['pred']=full['id'].map(preds)
_missing=full['pred'].isna().sum()
print('mapped ' + str(len(full)-_missing) + '/' + str(len(full)) + ' (missing=' + str(_missing) + ')')
assert _missing==0, 'ids unscored - run another session to resume (progress is on HF)'
for c in SUB_COLS: full[c]=(full['pred']==c).astype(int)
assert ((full[SUB_COLS].sum(axis=1))==1).all(), 'rows not one-hot!'
sub=full[['id']+SUB_COLS]
sub.to_csv(os.path.join(OUT_DIR,'submission.csv'),index=False)
print(sub.head(3).to_string()); print(sub[SUB_COLS].sum().to_dict())
_oks=_hf_push(os.path.join(OUT_DIR,'submission.csv'), HF_FINAL, 'submission 10k smolvlm adapter_best')
if _oks:
    print('submission.csv pushed -> ' + HF_REPO_ID + '/' + HF_FINAL)
else:
    print('SUBMISSION PUSH FAILED - download submission.csv from Kaggle output (' + OUT_DIR + ') and upload manually!')


submission_progress.csv:   0%|          | 0.00/169k [00:00<?, ?B/s]

scored rows from HF: 10000
mapped 10000/10000 (missing=0)
  id  same_figure  same_paper  related_papers  unrelated_papers
0  0            0           1               0                 0
1  1            0           1               0                 0
2  2            0           1               0                 0
{'same_figure': 905, 'same_paper': 6884, 'related_papers': 2211, 'unrelated_papers': 0}
submission.csv pushed -> ShivRamSaud/smolvlm-500m-lora-astroclimb/submission/submission.csv


In [9]:
# --- 7. Summary ---
print('test=' + str(len(test)) + ' scored=' + str(len(preds)) + ' skipped=' + str(len(skipped)) + ' oom=' + str(n_oom) + ' parse_fb=' + str(n_parse))
print('pred dist: ' + str(pd.Series(list(preds.values())).value_counts().to_dict()))
print('files live ONLY in ' + HF_REPO_ID + ': ' + HF_PROG + ' + ' + HF_RAW + ' + ' + HF_FINAL + ' (local staging deleted)')
print('Next: download submission.csv from the HF repo -> upload to competition')


test=10000 scored=10000 skipped=0 oom=0 parse_fb=0
pred dist: {'same_paper': 6884, 'related_papers': 2211, 'same_figure': 905}
files live ONLY in ShivRamSaud/smolvlm-500m-lora-astroclimb: submission/submission_progress.csv + submission/raw_test.jsonl + submission/submission.csv (local staging deleted)
Next: download submission.csv from the HF repo -> upload to competition
